# Liquidity Absorption V2 — CVD absorption filter, 18:00-ET volume-profile targets (LVN → next HVN), engine stop

Pre-registration: `discussions/Liquidity_Absorption/025_claude_v2_cvd_profile_preregistration.md`. Code: `src/data_engineering/modeling/liquidity_absorption_v2.py`.

**Reuses the EXT run:** its A02/A02b signals (`notebooks/Liquidity_Absorption/EXT/`) and venue inputs (`DERIBIT_DATA_ROOT\la_ext_venue`). Run EXT first.

**Primary:** CVD filter on, TRAIL_1M (breakeven at +1R, then 1-minute swing trail), profile target, VD18 risk stop, Deribit fees, the LA 024 gate.

**Also reported:** FIXED / BREAKEVEN, the same without the CVD filter, the CUT15 variants, and Kelly fractions on net R multiples.

`LAV2_SYNTHETIC=1` for a smoke run.

In [ ]:
from pathlib import Path
import json, os, sys, tempfile, time, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from dotenv import load_dotenv
warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 250)
load_dotenv()
REPO = Path.cwd().resolve()
while REPO != REPO.parent and not (REPO / 'src' / 'data_engineering').exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO)); sys.path.insert(0, str(REPO / 'src'))
from src.data_engineering.modeling import liquidity_absorption_v2 as v2, liquidity_absorption_extended as ex
SYN = os.getenv('LAV2_SYNTHETIC', '').strip() == '1'
if SYN:
    TMP = Path(tempfile.mkdtemp(prefix='lav2_')); ROOT = TMP / 'venue'; EXT = TMP / 'EXT'; OUT = TMP / 'V2'
    days = [str(x.date()) for x in pd.date_range('2026-06-01', periods=14, freq='D')]
    ex.synthetic_history(TMP / 'history', days, seed=6); ex.prepare_venue_root(TMP / 'history', ROOT, ex.Spec(start='2026-06-01'), log=None)
    ex.stage_a02(ROOT, EXT, ex.Spec(start='2026-06-04')); ex.stage_a02b(ROOT, EXT)
else:
    DATA = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
    ROOT = DATA / 'la_ext_venue'; EXT = REPO / 'notebooks' / 'Liquidity_Absorption' / 'EXT'; OUT = REPO / 'notebooks' / 'Liquidity_Absorption' / 'V2'
OUT.mkdir(parents=True, exist_ok=True)
print(v2.VERSION, '| synthetic' if SYN else '', '| EXT', EXT, '| out', OUT)
print(json.dumps(v2.registration(), indent=1))

## 1. Signals: CVD filter, session profile and LVN → HVN targets, VD18 sigma; simulation (6 variants per signal)

In [ ]:
t0 = time.time()
RES = v2.run_v2(EXT, ROOT, REPO, OUT)
SIG = pd.read_csv(OUT / 'V2_signals.csv', low_memory=False)
print('signals', len(SIG), '| CVD absorption share', round(SIG.cvd_ok.mean(), 3), f'| {time.time()-t0:.0f}s')
display(SIG.groupby(['symbol', 'cvd_ok']).size().unstack(fill_value=0))
display(SIG.v2_target_source.value_counts())
ok = SIG.v2_target.notna()
SIG['target_bp'] = np.abs(SIG.v2_target / SIG.bar_close - 1) * 1e4
display(SIG.loc[ok, ['target_bp', 'sigma_bp_60', 'session_age_min']].describe().round(1))

## 2. Verdict (primary) and all variants

In [ ]:
T, DEC = v2.evaluate(OUT)
T.to_csv(OUT / 'V2_results_by_period.csv', index=False)
(OUT / 'V2_decision.json').write_text(json.dumps(DEC, indent=1, default=str)); print(json.dumps(DEC, indent=1, default=str))
display(T[T.cvd_filter & T.variant.eq('TRAIL_1M')].round(3))
display(T[T.period.eq('HOLDOUT') & T.symbol.eq('ALL')].sort_values(['cvd_filter', 'variant']).round(3))

## 3. Does the CVD filter separate outcomes? (holdout, closed trades, descriptive)

In [ ]:
ev = pd.read_csv(OUT / 'V2_events.csv', low_memory=False); ev['net_deribit'] = ex.deribit_net(ev)
z = ev[ev.reason.isin(v2.CLOSED) & ev.v2_variant.eq('FIXED')].copy(); z['period'] = ex.period_label(z.signal_time_utc)
z = z[z.period != 'INSAMPLE_JUL_SEP_2026']
display(z.groupby(['symbol', 'cvd_ok']).agg(n=('net_deribit', 'size'), target_rate=('reason', lambda s: (s == 'TARGET').mean()),
        mean_gross=('gross_return_bp', 'mean'), mean_net=('net_deribit', 'mean'), median_rr=('reward_to_risk', 'median')).round(3))

## 4. Equity, primary variant (Deribit fees, 1 unit per trade, holdout and in-sample shaded)

In [ ]:
p = ev[ev.cvd_ok & ev.v2_variant.eq('TRAIL_1M') & ev.reason.isin(v2.CLOSED)].copy()
p['t'] = pd.to_datetime(p.exit_time_ms, unit='ms', utc=True); p = p.sort_values('t')
fig, ax = plt.subplots(figsize=(14, 5))
for s, g in p.groupby('symbol'):
    ax.plot(g.t, g.net_deribit.cumsum() / 100, label=f'{s} net'); ax.plot(g.t, g.gross_return_bp.cumsum() / 100, ls='--', label=f'{s} gross')
ax.axvspan(*ex.INSAMPLE, color='grey', alpha=.2, label='in-sample Jul-Sep 2026'); ax.axhline(0, c='k', lw=.5); ax.legend()
ax.set_title('V2 primary: cumulative % of notional per trade'); plt.tight_layout(); plt.savefig(OUT / 'V2_equity.png', dpi=110); plt.show()